# Dataset Setup (SpeechBCI Preprocessed Dataset)

This project uses a preprocessed SpeechBCI dataset (~3.6 GB).
Because of size limits, it is not stored in this GitHub repo.

Please download the dataset from the following Google Drive Link and mount your drive to be able to run the demo.

Dataset Google Drive Source:  
https://drive.google.com/file/d/15t8IDF4CUsXtMu3xurs48TSYjyH_BsVZ/view?usp=sharing


# Run Once To Install WER Library

In [2]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 117.5 MB/s eta 0:00:00


# Run Once to Mount to Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')
%cd "PATH TO .pkl FILE"

Mounted at /content/drive
/content/drive/MyDrive/Fall 2025 UCLA/ECE_243A/Final Project/neural_seq_decoder/scripts


# Phoneme to Text Demo Code

In [6]:
import torch
import numpy as np
import pickle
import random
import string
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from jiwer import wer

###############################################
# Phoneme Definition
###############################################

PHONE_DEF = [
    'AA', 'AE', 'AH', 'AO', 'AW',
    'AY', 'B',  'CH', 'D', 'DH',
    'EH', 'ER', 'EY', 'F', 'G',
    'HH', 'IH', 'IY', 'JH', 'K',
    'L', 'M', 'N', 'NG', 'OW',
    'OY', 'P', 'R', 'S', 'SH',
    'T', 'TH', 'UH', 'UW', 'V',
    'W', 'Y', 'Z', 'ZH'
]
PHONE_DEF_SIL = PHONE_DEF + ['SIL']


###############################################
# Helper Functions
###############################################

def index_to_phoneme_sequence(idx_list):
    phoneme_text = ""
    for index in idx_list:
        if index == 0:
            break
        phoneme = PHONE_DEF_SIL[index - 1]
        phoneme_text += f'{phoneme} '
    return phoneme_text.strip()


def decode_from_indices(
    phoneme_sequence: str,
    hf_tokenizer,
    hf_model,
    device: str
) -> str:

    input_text = "Transcribe phonemes to standard English: " + phoneme_sequence

    input_ids = hf_tokenizer(
        input_text,
        return_tensors="pt",
        padding='max_length',
        max_length=256
    ).input_ids.to(device)

    predicted_ids = hf_model.generate(
        input_ids,
        max_length=128,
        num_beams=5,
        early_stopping=True,
        do_sample=False,
    )

    predicted_text = hf_tokenizer.decode(
        predicted_ids.squeeze(),
        skip_special_tokens=True
    )

    return predicted_text


def clean_text_for_wer(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text


###############################################
# Single Example Decoding Pipeline
###############################################

def run_single_sample(index=0):
    MODEL_REPO_ID = "tonykorol/t5_phoneme_decoder"

    # Update this path if needed when running outside Colab
    DATA_PATH = "PATH TO .pkl FILE"

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    # Load HF model + tokenizer
    hf_tokenizer = AutoTokenizer.from_pretrained("t5-small")
    hf_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_REPO_ID).to(device)

    # Load dataset
    with open(DATA_PATH, 'rb') as f:
        all_datasets = pickle.load(f)

    test_dataset = all_datasets['test']
    test_dataset.extend(all_datasets['train'])

    if index >= len(test_dataset) or index < 0:
        raise ValueError(f"Index out of range. Dataset size = {len(test_dataset)}")

    sample = test_dataset[index]
    raw_phoneme_indices = np.array(sample['phonemes']).flatten().tolist()
    ground_truth_text = sample['transcriptions'][0]

    # Convert indices → phoneme text
    phoneme_seq = index_to_phoneme_sequence(raw_phoneme_indices)

    # Decode using T5
    decoded_text = decode_from_indices(
        phoneme_sequence=phoneme_seq,
        hf_tokenizer=hf_tokenizer,
        hf_model=hf_model,
        device=device
    )

    # Compute WER
    sample_wer = wer(
        reference=clean_text_for_wer(ground_truth_text),
        hypothesis=clean_text_for_wer(decoded_text)
    )

    ###############################################
    # Display Results
    ###############################################
    print("\n========================================")
    print(f" Sample Index: {index}")
    print("========================================\n")

    print("Phoneme Input:")
    print(f"Transcribe phonemes to standard English: {phoneme_seq}\n")

    print("Ground Truth Text:")
    print(ground_truth_text, "\n")

    print("Decoded Text:")
    print(decoded_text, "\n")

    print(f"WER: {sample_wer:.4f}")
    print("\n========================================\n")


###############################################
# Run Example
###############################################
# Change this to test different datapoints
run_single_sample(index=5)


Using device: cuda

 Sample Index: 5

Phoneme Input:
Transcribe phonemes to standard English: DH IH S SIL IH Z SIL Y UW N AH F AY IH NG SIL V OY S SIL W IY SIL N IY D SIL

Ground Truth Text:
This is unifying voice we need. 

Decoded Text:
This is inspiring voice we need. 

WER: 0.1667


